In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
from gensim.models import KeyedVectors
from collections import Counter

## 02a - Prepare Outfits

Filtra outfits con mas de 8 prendas, verifica imagenes y metadatos, y construye el indice global de items.

**Salidas en `./processed/`:**  
`outfits_train.json` / `outfits_valid.json` / `outfits_test.json` / `item_index.json`

### Configuracion -- edita estas rutas antes de ejecutar

In [ ]:
# EDITA ESTAS RUTAS
DATA_ROOT = Path('./polyvore_kaggle')
MAX_ITEMS = 8  # maximo prendas por outfit

OUT_DIR   = DATA_ROOT / 'polyvore_outfits' / 'data' / 'processed'
OUTFITS_DIR = DATA_ROOT / 'polyvore_outfits' / 'nondisjoint'
IMG_DIR     = DATA_ROOT / 'polyvore_outfits' / 'images'
META_PATH   = DATA_ROOT / 'polyvore_outfits' / 'polyvore_item_metadata.json'

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATA_ROOT : {DATA_ROOT.resolve()}')
print(f'OUT_DIR   : {OUT_DIR.resolve()}')
print(f'MAX_ITEMS : {MAX_ITEMS}')

### 1. Verificar estructura de archivos

In [ ]:
required = {
    'Imagenes':                    IMG_DIR,
    'train.json (disjoint)':       OUTFITS_DIR / 'train.json',
    'valid.json (disjoint)':       OUTFITS_DIR / 'valid.json',
    'test.json  (disjoint)':       OUTFITS_DIR / 'test.json',
    'compatibility_train.txt':     OUTFITS_DIR / 'compatibility_train.txt',
    'compatibility_valid.txt':     OUTFITS_DIR / 'compatibility_valid.txt',
    'compatibility_test.txt':      OUTFITS_DIR / 'compatibility_test.txt',
    'fill_in_blank_test.json':     OUTFITS_DIR / 'fill_in_blank_test.json',
    'polyvore_item_metadata.json': META_PATH,
}

all_ok = True
for name, path in required.items():
    mark = 'OK' if path.exists() else 'FALTA'
    print(f'  {mark}  {name:<40}  {path}')
    if not path.exists():
        all_ok = False

print()
print('Todos los archivos encontrados' if all_ok else 'ERROR: faltan archivos -- revisa DATA_ROOT')

### 2. Cargar metadatos y analizar cobertura de texto

In [ ]:
with open(META_PATH) as f:
    metadata = json.load(f)

print(f'Total items: {len(metadata):,}')
print(f'Campos    : {list(next(iter(metadata.values())).keys())}\n')

N = len(metadata)
fields = ['url_name', 'title', 'description']
print(f'{"Campo":<15} {"Con valor":>10} {"%":>7}  {"Palabras (media)":>16}')
print('-' * 52)
for field in fields:
    vals  = [v.get(field, '').strip() for v in metadata.values()]
    n     = sum(1 for v in vals if v)
    words = [len(v.split()) for v in vals if v]
    avg   = sum(words) / len(words) if words else 0
    print(f'  {field:<13} {n:>10,} {100*n/N:>7.1f}%  {avg:>16.1f}')

print()
print('Estrategia de texto aplicada:')
print('  texto = description  (mayor riqueza semantica cuando existe)')
print('  texto += title       (si no es redundante con description)')
print('  texto += url_name    (siempre -- garantiza 100% cobertura)')


### 3. Verificar que los splits son disjoint

In [ ]:
ids = {}
for split in ['train', 'valid', 'test']:
    with open(OUTFITS_DIR / f'{split}.json') as f:
        outfits = json.load(f)
    ids[split] = set(it['item_id'] for o in outfits for it in o['items'])
    print(f'  {split:6s}: {len(ids[split]):,} items unicos')

print()
for a, b in [('train','valid'), ('train','test'), ('valid','test')]:
    overlap = ids[a] & ids[b]
    mark = 'OK' if not overlap else 'SOLAPAMIENTO'
    msg  = 'conjunto vacio (correcto)' if not overlap else f'{len(overlap)} items solapados'
    print(f'  {mark}  {a} interseccion {b} = {msg}')


### 4. Filtrar outfits y construir item_index

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

OUT_DIR   = Path('./polyvore_kaggle/polyvore_outfits/data/processed')
MAX_ITEMS = 8

ds_splits = {
    'train': load_dataset('owj0421/polyvore-outfits', 'nondisjoint_default', split='train'),
    'valid': load_dataset('owj0421/polyvore-outfits', 'nondisjoint_default', split='validation'),
    'test' : load_dataset('owj0421/polyvore-outfits', 'nondisjoint_default', split='test'),
}

for split_name, ds in ds_splits.items():
    outfits = []
    for outfit in tqdm(ds, desc=f'Procesando {split_name}'):
        if len(outfit['items']) > MAX_ITEMS:
            continue
        outfits.append({
            'set_id': str(outfit['set_id']),
            'items' : [{'item_id': str(it['item_id']), 'index': it['index']}
                       for it in outfit['items']]
        })

    with open(OUT_DIR / f'outfits_{split_name}.json', 'w') as f:
        json.dump(outfits, f)
    print(f'  {split_name}: {len(outfits):,} outfits guardados')

In [ ]:
# item_index sigue siendo necesario para las celdas siguientes
with open(OUT_DIR / 'item_index_full_nondisjoint.json') as f:
    item_index = {str(k): v for k, v in json.load(f).items()}
print(f'item_index cargado: {len(item_index):,} items')

In [ ]:
# def build_text(meta):
#     desc  = meta.get('description', '').strip()
#     title = meta.get('title', '').strip()
#     url   = meta.get('url_name', '').strip().replace('-', ' ')
#     parts = []
#     if desc:  parts.append(desc)
#     if title and title.lower() not in desc.lower(): parts.append(title)
#     if url:   parts.append(url)
#     return ' '.join(parts)

# item_index = {}
# stats = {}

# for split in ['train', 'valid', 'test']:
#     with open(OUTFITS_DIR / f'{split}.json') as f:
#         raw = json.load(f)

#     kept = []
#     skip_size = skip_img = skip_meta = 0

#     for outfit in raw:
#         items_raw = outfit.get('items', [])
#         if len(items_raw) > MAX_ITEMS:
#             skip_size += 1
#             continue

#         items_clean, ok = [], True
#         for item in items_raw:
#             iid      = str(item['item_id'])
#             img_path = IMG_DIR / f'{iid}.jpg'
#             if not img_path.exists():
#                 skip_img += 1; ok = False; break
#             if iid not in metadata:
#                 skip_meta += 1; ok = False; break

#             m        = metadata[iid]
#             text     = build_text(m)
#             category = str(m.get('semantic_category', m.get('category_id', 'unknown')))

#             items_clean.append({
#                 'item_id':    iid,
#                 'image_path': str(img_path.relative_to(DATA_ROOT)),
#                 'text':       text,
#                 'category':   category,
#                 'index':      item.get('index', 0),
#             })
#             if iid not in item_index:
#                 item_index[iid] = {
#                     'image_path': str(img_path.relative_to(DATA_ROOT)),
#                     'text': text, 'category': category
#                 }
#         if ok:
#             kept.append({'set_id': str(outfit.get('set_id', '')), 'items': items_clean})

#     stats[split] = {'raw': len(raw), 'kept': len(kept),
#                     'skip_size': skip_size, 'skip_img': skip_img, 'skip_meta': skip_meta}
#     with open(OUT_DIR / f'outfits_{split}.json', 'w') as f:
#         json.dump(kept, f)

# with open(OUT_DIR / 'item_index.json', 'w') as f:
#     json.dump(item_index, f)

# print(f'{"Split":<8} {"Raw":>8} {"Kept":>8} {"%kept":>7}  {">8 prendas":>11} {"Sin img":>8} {"Sin meta":>9}')
# print('-' * 60)
# for split, s in stats.items():
#     pct = 100 * s['kept'] / s['raw']
#     print(f'  {split:<6} {s["raw"]:>8,} {s["kept"]:>8,} {pct:>7.1f}%  '
#           f'{s["skip_size"]:>11,} {s["skip_img"]:>8,} {s["skip_meta"]:>9,}')

# print(f'\nitem_index.json: {len(item_index):,} items unicos')

### 5. Distribucion de categorias

In [ ]:
import matplotlib.pyplot as plt

cat_counter = Counter(v['category'] for v in item_index.values())
top15 = cat_counter.most_common(15)

fig, ax = plt.subplots(figsize=(10, 5))
cats, cnts = zip(*top15)
ax.barh(list(cats)[::-1], list(cnts)[::-1], color='#4C72B0', edgecolor='white')
ax.set_xlabel('Numero de items')
ax.set_title(f'Top 15 categorias -- {len(cat_counter)} categorias unicas')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 6. Distribucion de longitud de outfits

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, split in zip(axes, ['train', 'valid', 'test']):
    with open(OUT_DIR / f'outfits_{split}.json') as f:
        outfits = json.load(f)
    lengths = [len(o['items']) for o in outfits]
    cnt = Counter(lengths)
    ax.bar(cnt.keys(), cnt.values(), color='#55A868', edgecolor='white')
    ax.set_title(f'{split.capitalize()} ({len(outfits):,} outfits)')
    ax.set_xlabel('Prendas por outfit')
    ax.set_ylabel('Frecuencia')
    ax.set_xticks(range(1, MAX_ITEMS + 1))
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Distribucion de prendas por outfit', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


### 7. Resumen final

In [ ]:
print('Archivos generados en ./processed/\n')
for split in ['train', 'valid', 'test']:
    with open(OUT_DIR / f'outfits_{split}.json') as f: data = json.load(f)
    n_items = sum(len(o['items']) for o in data)
    print(f'  outfits_{split}.json   -> {len(data):>6,} outfits  |  {n_items:>7,} items')

texts   = [v['text'] for v in item_index.values()]
n_empty = sum(1 for t in texts if not t.strip())
lengths = [len(t.split()) for t in texts if t.strip()]
print(f'  item_index.json        -> {len(item_index):>6,} items unicos')
print(f'\nCobertura de texto:')
print(f'  Con texto : {len(item_index)-n_empty:,}  ({100*(len(item_index)-n_empty)/len(item_index):.1f}%)')
print(f'  Sin texto : {n_empty:,}  ({100*n_empty/len(item_index):.1f}%)')
print(f'  Palabras/item (media): {np.mean(lengths):.1f}')
print('\nSiguiente paso: 02b_fisher_text_embeddings.ipynb')

## 02b - Fisher Text Embeddings

Genera embeddings de texto para cada item usando el pipeline del paper SCE-Net:

`texto` -> `word2vec (300d)` -> `GMM diagonal (HGLMM)` -> `Fisher Vector (2*K*D)` -> `PCA -> 6000 dims`

**Requisitos:**
```bash
pip install gensim scikit-learn numpy tqdm matplotlib
```
Descargar word2vec Google News (~1.5 GB):
https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM

**Salidas en `data/processed/`:**  
`fisher_vectors_nondisjoint.npy` / `fisher_item_ids_nondisjoint.json` / `gmm_model_nondisjoint.pkl` / `pca_model_nondisjoint.pkl` / `text_stats_nondisjoint.json`

### Configuracion

In [ ]:
# EDITA ESTAS RUTAS
# OUT_DIR      = Path('./processed')
# META_PATH    = Path('./polyvore_kaggle/polyvore_outfits/polyvore_item_metadata.json')
WORD2VEC_BIN = Path('./polyvore_kaggle/polyvore_outfits/GoogleNews-vectors-negative300.bin/GoogleNews-vectors-negative300.bin')  # <- EDITA ESTO

# Hiperparametros (paper: K=64, pca_dim=6000)
K               = 64
PCA_DIM         = 2048  # Esto exige 20GB de RAM, vamos a usar 2048 y BATCHSIZE = 4096
MAX_WORDS       = 64
MIN_FREQ        = 2
MAX_GMM_SAMPLES = 500_000
SEED            = 42

np.random.seed(SEED)

print(f'OUT_DIR      : {OUT_DIR.resolve()}')
print(f'WORD2VEC_BIN : {WORD2VEC_BIN}  {"OK" if WORD2VEC_BIN.exists() else "NO ENCONTRADO"}')
print(f'K={K}  PCA_DIM={PCA_DIM}  MAX_WORDS={MAX_WORDS}')


### 1. Cargar item_index y metadatos

In [ ]:
import json

with open(OUT_DIR / 'item_index_full_nondisjoint.json') as f:
    item_index = json.load(f)
with open(META_PATH) as f:
    metadata = json.load(f)

item_ids = list(item_index.keys())
print(f'Items a procesar : {len(item_ids):,}')
print(f'Metadatos totales: {len(metadata):,}')


### 2. Construccion del texto final por item

Estrategia hibrida basada en la cobertura real:
- `description` (28.7%, ~38 palabras) -- maxima riqueza semantica
- `title` (29.8%, ~6 palabras) -- complementario si no es redundante
- `url_name` (100%, ~4.6 palabras) -- ancla que garantiza cobertura total

In [ ]:
import re

def build_text(item_id):
    meta  = metadata.get(str(item_id), {})
    desc  = meta.get('description', '').strip()
    title = meta.get('title', '').strip()
    url   = meta.get('url_name', '').strip().replace('-', ' ')
    parts = []
    if desc:  parts.append(desc)
    if title and title.lower() not in desc.lower(): parts.append(title)
    if url:   parts.append(url)
    return ' '.join(parts)

STOPWORDS = {
    'a','an','the','and','or','of','to','in','for',
    'with','is','are','was','were','this','that','from',
    'at','by','on','be','as','it','its','has','have',
    'all','also','can','will','not','but','so','if'
}

def tokenize(text):
    tokens = re.findall(r'[a-z]+', text.lower())   # solo letras, sin numeros
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens[:MAX_WORDS]

texts  = [build_text(iid) for iid in item_ids]
tokens = [tokenize(t) for t in texts]

lengths = [len(t) for t in tokens]
n_empty = sum(1 for t in tokens if not t)
n_desc  = sum(1 for iid in item_ids
              if metadata.get(str(iid), {}).get('description', '').strip())

print(f'Items con tokens : {len(item_ids)-n_empty:,} / {len(item_ids):,}  ({100*(len(item_ids)-n_empty)/len(item_ids):.1f}%)')
print(f'  Con description: {n_desc:,}  ({100*n_desc/len(item_ids):.1f}%)')
print(f'  Solo url/title : {len(item_ids)-n_empty-n_desc:,}')
print(f'  Sin texto      : {n_empty:,}')
print(f'Tokens/item media: {np.mean(lengths):.1f}  min: {min(lengths)}  max: {max(lengths)}')

print('\nEjemplos:')
shown = {'desc': False, 'url': False}
for iid, text in zip(item_ids, texts):
    has_desc = bool(metadata.get(str(iid), {}).get('description', '').strip())
    if has_desc and not shown['desc']:
        print(f'\n  [Con description]  {iid}')
        print(f'  {text[:120]}')
        shown['desc'] = True
    elif not has_desc and text and not shown['url']:
        print(f'\n  [Solo url/title]   {iid}')
        print(f'  {text[:120]}')
        shown['url'] = True
    if all(shown.values()): break

## 3. Cargar word2vec

In [ ]:
print('Cargando word2vec... (puede tardar ~1 min)')
w2v     = KeyedVectors.load_word2vec_format(str(WORD2VEC_BIN), binary=True)
W2V_DIM = w2v.vector_size
print(f'word2vec: {len(w2v.key_to_index):,} palabras, dim={W2V_DIM}')

word_freq     = Counter(tok for tl in tokens for tok in tl)
dataset_vocab = {w for w, c in word_freq.items() if c >= MIN_FREQ}
in_w2v        = sum(1 for w in dataset_vocab if w in w2v.key_to_index)
print(f'Vocabulario dataset (freq>={MIN_FREQ}): {len(dataset_vocab):,}')
print(f'Cobertura en word2vec: {in_w2v:,}  ({100*in_w2v/max(len(dataset_vocab),1):.1f}%)')

In [ ]:
# Añade esta celda al 02b para diagnóstico
missing = {w for w in dataset_vocab if w not in w2v.key_to_index}

# Ver los más frecuentes de los que faltan
missing_freq = {w: word_freq[w] for w in missing}
top_missing  = sorted(missing_freq.items(), key=lambda x: -x[1])[:30]

print("Top 30 palabras del dataset NO cubiertas por word2vec:")
for word, freq in top_missing:
    print(f"  {word:30s}  freq={freq:,}")

### 4. Vectorizar items

In [ ]:
from tqdm.notebook import tqdm

def get_vecs(tok_list):
    v = [w2v[t] for t in tok_list if t in w2v.key_to_index]
    return np.array(v, dtype=np.float32) if v else None

item_vecs = [get_vecs(tl) for tl in tqdm(tokens, desc='Vectorizando')]

n_with = sum(1 for v in item_vecs if v is not None)
print(f'\nItems con vectores w2v: {n_with:,} / {len(item_ids):,}  ({100*n_with/len(item_ids):.1f}%)')

### 5. Entrenar GMM (aproximacion HGLMM diagonal)

El paper usa HGLMM (Klein et al., CVPR 2015): una GMM diagonal con optimizacion Newton-Raphson sobre word2vec. Usamos `GaussianMixture(covariance_type='diag')` de sklearn, que es una aproximacion matematicamente equivalente.

In [ ]:
from sklearn.mixture import GaussianMixture
import pickle, warnings
warnings.filterwarnings('ignore')

all_vecs = np.vstack([v for v in item_vecs if v is not None])
print(f'Vectores disponibles: {len(all_vecs):,} x {W2V_DIM}')

if len(all_vecs) > MAX_GMM_SAMPLES:
    print(f'Submuestreando a {MAX_GMM_SAMPLES:,} vectores...')
    idx      = np.random.choice(len(all_vecs), MAX_GMM_SAMPLES, replace=False)
    gmm_data = all_vecs[idx]
else:
    gmm_data = all_vecs

print(f'Entrenando GMM K={K}... (puede tardar 10-30 min)\n')

gmm = GaussianMixture(
    n_components=K,
    covariance_type='diag',
    max_iter=200,
    n_init=1,
    verbose=2,
    random_state=SEED,
    reg_covar=1e-3,   # <- añadir esto
)
gmm.fit(gmm_data.astype(np.float64))   # <- float64 en lugar de float32

with open(OUT_DIR / 'gmm_model_nondisjoint.pkl', 'wb') as f:
    pickle.dump(gmm, f)

print(f'\nConverged: {gmm.converged_}  lower_bound: {gmm.lower_bound_:.6f}')
print(f'Guardado -> {OUT_DIR}/gmm_model_nondisjoint.pkl')


### 6. Calcular Fisher Vectors

Derivadas analiticas del log-likelihood respecto a medias y varianzas:

- **G_mu[k]** = (1 / T*sqrt(pi_k)) * sum_t gamma(t,k) * (x_t - mu_k) / sigma_k  
- **G_sigma[k]** = (1 / T*sqrt(2*pi_k)) * sum_t gamma(t,k) * [(x_t-mu_k)^2/sigma_k - 1]

Normalizacion: power norm (signed sqrt) seguido de L2 norm.

In [ ]:
# K_      = gmm.n_components
# D_      = W2V_DIM
# FV_DIM  = 2 * K_ * D_
# _mu     = gmm.means_
# _sigma  = gmm.covariances_
# _pi     = gmm.weights_
# _log_pi = np.log(_pi + 1e-10)

# print(f'Dimension Fisher Vector cruda: 2 x {K_} x {D_} = {FV_DIM:,}')

# def compute_fisher_vector(word_vecs):
#     T    = len(word_vecs)
#     diff = word_vecs[:, None, :] - _mu[None, :, :]    # (T, K, D)

#     log_lik   = -0.5 * (
#         np.log(2 * np.pi) +
#         np.log(_sigma + 1e-10)[None, :, :] +
#         diff**2 / (_sigma + 1e-10)[None, :, :]
#     ).sum(axis=2)                                      # (T, K)

#     log_gamma  = log_lik + _log_pi[None, :]
#     log_gamma -= log_gamma.max(axis=1, keepdims=True)
#     gamma      = np.exp(log_gamma)
#     gamma     /= gamma.sum(axis=1, keepdims=True) + 1e-10

#     sqrt_pi = np.sqrt(_pi + 1e-10)

#     G_mu = (
#         (gamma[:, :, None] * diff).sum(axis=0)
#         / (_sigma + 1e-10)
#         / (T * sqrt_pi[:, None])
#     )
#     G_sigma = (
#         (gamma[:, :, None] * (diff**2 / (_sigma + 1e-10) - 1)).sum(axis=0)
#         / (np.sqrt(2) * T * sqrt_pi[:, None])
#     )

#     fv   = np.concatenate([G_mu.ravel(), G_sigma.ravel()])
#     fv   = np.sign(fv) * np.sqrt(np.abs(fv) + 1e-10)  # power norm
#     norm = np.linalg.norm(fv)
#     if norm > 1e-10: fv /= norm
#     return fv.astype(np.float32)

# fisher_raw = np.zeros((len(item_ids), FV_DIM), dtype=np.float32)
# n_zero     = 0

# for i, vecs in enumerate(tqdm(item_vecs, desc='Fisher Vectors')):
#     if vecs is not None and len(vecs) > 0:
#         fisher_raw[i] = compute_fisher_vector(vecs)
#     else:
#         n_zero += 1

# print(f'\nFisher Vectors: {fisher_raw.shape}')
# print(f'Items sin vector (zeros): {n_zero:,}')
# print(f'NaN en matriz           : {np.isnan(fisher_raw).sum()}')
# print(f'Norma L2 media          : {np.linalg.norm(fisher_raw, axis=1).mean():.4f}')


In [ ]:
import tempfile, os

K_      = gmm.n_components
D_      = W2V_DIM
FV_DIM  = 2 * K_ * D_
_mu     = gmm.means_
_sigma  = gmm.covariances_
_pi     = gmm.weights_
_log_pi = np.log(_pi + 1e-10)

print(f'Dimension Fisher Vector cruda: 2 x {K_} x {D_} = {FV_DIM:,}')

def compute_fisher_vector(word_vecs):
    T    = len(word_vecs)
    diff = word_vecs[:, None, :] - _mu[None, :, :]
    log_lik   = -0.5 * (
        np.log(2 * np.pi) +
        np.log(_sigma + 1e-10)[None, :, :] +
        diff**2 / (_sigma + 1e-10)[None, :, :]
    ).sum(axis=2)
    log_gamma  = log_lik + _log_pi[None, :]
    log_gamma -= log_gamma.max(axis=1, keepdims=True)
    gamma      = np.exp(log_gamma)
    gamma     /= gamma.sum(axis=1, keepdims=True) + 1e-10
    sqrt_pi    = np.sqrt(_pi + 1e-10)
    G_mu = (
        (gamma[:, :, None] * diff).sum(axis=0)
        / (_sigma + 1e-10)
        / (T * sqrt_pi[:, None])
    )
    G_sigma = (
        (gamma[:, :, None] * (diff**2 / (_sigma + 1e-10) - 1)).sum(axis=0)
        / (np.sqrt(2) * T * sqrt_pi[:, None])
    )
    fv   = np.concatenate([G_mu.ravel(), G_sigma.ravel()])
    fv   = np.sign(fv) * np.sqrt(np.abs(fv) + 1e-10)
    norm = np.linalg.norm(fv)
    if norm > 1e-10: fv /= norm
    return fv.astype(np.float32)

# Guardar en disco usando memmap — nunca carga todo en RAM
fv_raw_path = OUT_DIR / 'fisher_raw_memmap.npy'
fisher_raw  = np.lib.format.open_memmap(
    fv_raw_path, mode='w+',
    dtype=np.float32,
    shape=(len(item_ids), FV_DIM)
)

n_zero = 0
for i, vecs in enumerate(tqdm(item_vecs, desc='Fisher Vectors')):
    if vecs is not None and len(vecs) > 0:
        fisher_raw[i] = compute_fisher_vector(vecs)
    else:
        n_zero += 1
fisher_raw.flush()

print(f'\nFisher Vectors escritos en disco: {fv_raw_path}')
print(f'Shape  : {fisher_raw.shape}')
print(f'Sin vector (zeros): {n_zero:,}')

### 7. PCA

In [ ]:
# from sklearn.decomposition import IncrementalPCA

# actual_pca_dim = min(PCA_DIM, FV_DIM)

# # IncrementalPCA procesa en batches, evita cargar toda la matriz en RAM
# BATCH_SIZE = 4098  # ajusta segun tu RAM disponible

# pca = IncrementalPCA(n_components=actual_pca_dim, batch_size=BATCH_SIZE)

# # Ajustar solo sobre filas no-zero
# nonzero_idx = np.where(np.linalg.norm(fisher_raw, axis=1) > 1e-10)[0]
# print(f'Ajustando IncrementalPCA ({actual_pca_dim} componentes) '
#       f'sobre {len(nonzero_idx):,} filas en batches de {BATCH_SIZE}...')

# for start in tqdm(range(0, len(nonzero_idx), BATCH_SIZE), desc='PCA fit'):
#     batch_idx = nonzero_idx[start:start + BATCH_SIZE]
#     pca.partial_fit(fisher_raw[batch_idx].astype(np.float64))

# var_exp = pca.explained_variance_ratio_.sum()
# print(f'Varianza explicada: {100*var_exp:.1f}%')

# # Transformar en batches tambien
# print('Transformando...')
# fisher_pca = np.zeros((len(item_ids), actual_pca_dim), dtype=np.float32)
# for start in tqdm(range(0, len(fisher_raw), BATCH_SIZE), desc='PCA transform'):
#     end = min(start + BATCH_SIZE, len(fisher_raw))
#     fisher_pca[start:end] = pca.transform(
#         fisher_raw[start:end].astype(np.float64)
#     ).astype(np.float32)

# # Re-normalizar
# norms      = np.linalg.norm(fisher_pca, axis=1, keepdims=True)
# fisher_pca = fisher_pca / np.where(norms > 1e-10, norms, 1.0)

# print(f'Fisher Vectors finales: {fisher_pca.shape}')

In [ ]:
from sklearn.decomposition import IncrementalPCA

actual_pca_dim = min(PCA_DIM, FV_DIM)
BATCH_SIZE     = 4098   # batches pequeños para no saturar RAM

pca = IncrementalPCA(n_components=actual_pca_dim, batch_size=BATCH_SIZE)

# Fit en batches — solo lee BATCH_SIZE filas a la vez
print(f'Ajustando PCA ({actual_pca_dim} dims) en batches de {BATCH_SIZE}...')
for start in tqdm(range(0, len(item_ids), BATCH_SIZE), desc='PCA fit'):
    end   = min(start + BATCH_SIZE, len(item_ids))
    batch = np.array(fisher_raw[start:end], dtype=np.float64)
    # Saltar batches de puro zeros
    if np.linalg.norm(batch) < 1e-10:
        continue
    pca.partial_fit(batch)

var_exp = pca.explained_variance_ratio_.sum()
print(f'Varianza explicada: {100*var_exp:.1f}%')

# Transform en batches
print('Transformando...')
fisher_pca = np.zeros((len(item_ids), actual_pca_dim), dtype=np.float32)
for start in tqdm(range(0, len(item_ids), BATCH_SIZE), desc='PCA transform'):
    end = min(start + BATCH_SIZE, len(item_ids))
    fisher_pca[start:end] = pca.transform(
        np.array(fisher_raw[start:end], dtype=np.float64)
    ).astype(np.float32)

# Re-normalizar
norms      = np.linalg.norm(fisher_pca, axis=1, keepdims=True)
fisher_pca = fisher_pca / np.where(norms > 1e-10, norms, 1.0)

print(f'Fisher Vectors finales: {fisher_pca.shape}')

# Guardar PCA
import pickle
with open(OUT_DIR / 'pca_model_nondisjoint.pkl', 'wb') as f:
    pickle.dump(pca, f)
print(f'pca_model_nondisjoint.pkl guardado -> {OUT_DIR / "pca_model_nondisjoint.pkl"}')

# Limpiar memmap temporal
del fisher_raw
os.remove(fv_raw_path)
print('Memmap temporal eliminado')

### 8. Curva de varianza explicada

In [ ]:
import matplotlib.pyplot as plt

cumvar = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cumvar, color='#4C72B0', linewidth=1.5)
ax.axhline(0.9,  color='orange', linestyle='--', label='90%')
ax.axhline(0.95, color='red',    linestyle='--', label='95%')
ax.set_xlabel('Numero de componentes PCA')
ax.set_ylabel('Varianza explicada acumulada')
ax.set_title(f'PCA sobre Fisher Vectors ({FV_DIM:,} -> {actual_pca_dim} dims)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for threshold in [0.80, 0.90, 0.95]:
    n = int(np.searchsorted(cumvar, threshold)) + 1
    print(f'  Componentes para {100*threshold:.0f}% varianza: {n:,}')


### 9. Guardar resultados

In [ ]:
np.save(OUT_DIR / 'fisher_vectors_nondisjoint.npy', fisher_pca)
with open(OUT_DIR / 'fisher_item_ids_nondisjoint.json', 'w') as f:
    json.dump(item_ids, f)

stats = {
    'n_items':         len(item_ids),
    'fv_dim_raw':      int(FV_DIM),
    'fv_dim_final':    int(actual_pca_dim),
    'gmm_k':           K,
    'w2v_dim':         int(W2V_DIM),
    'n_items_no_text': n_zero,
    'pct_with_desc':   round(100 * n_desc / len(item_ids), 2),
    'var_explained':   round(float(var_exp), 4),
    'text_strategy':   'description + title (no redundante) + url_name',
}
with open(OUT_DIR / 'text_stat_nondisjoint.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('Archivos guardados en ./processed/')
print(f'  fisher_vectors_nondisjoint.npy    {fisher_pca.shape}  float32')
print(f'  fisher_item_ids_nondisjoint.json  {len(item_ids):,} ids')
print(f'  gmm_model.pkl         K={K}')
print(f'  pca_model.pkl         {actual_pca_dim} dims  ({100*var_exp:.1f}% varianza)')
print(f'  text_stat_nondisjoint.json')

print('\nVerificacion (3 items aleatorios):')
for _ in range(3):
    i  = np.random.randint(len(item_ids))
    fv = fisher_pca[i]
    t  = texts[i][:70]
    print(f'  [{item_ids[i]}] {t}...')
    print(f'    norm={np.linalg.norm(fv):.4f}  nonzero={np.count_nonzero(fv)}/{actual_pca_dim}')

print('\nSiguiente: 02c_build_triplets.ipynb')

## Paso 02c -- Generacion de tripletas para SCE-Net

Genera tripletas `(anchor, positive, negative)` para entrenamiento:
- **anchor** (xi): item de referencia del outfit
- **positive** (xj): otro item del **mismo outfit** (compatible)
- **negative** (xk): item de un **outfit diferente** (incompatible)

Mezcla de negativos:
- **Hard negatives**: misma categoria, outfit diferente
- **Random negatives**: item aleatorio de otro outfit

**Salidas en `./processed/`:**  
`triplets_train.json` / `triplets_valid.json`

### Configuracion

In [ ]:
from pathlib import Path

OUT_DIR       = Path('./polyvore_kaggle/polyvore_outfits/data/processed')
N_NEG         = 1    # negativos por par positivo
HARD_NEG_FRAC = 0.8  # fraccion de hard negatives (0 a 1)
SEED          = 42

import json, random, numpy as np
from collections import Counter, defaultdict
random.seed(SEED)

print(f'OUT_DIR       : {OUT_DIR.resolve()}')
print(f'N_NEG         : {N_NEG}')
print(f'HARD_NEG_FRAC : {HARD_NEG_FRAC}')


### 1. Cargar item_index

In [ ]:
with open(OUT_DIR / 'item_index_full_nondisjoint.json') as f:
    item_index = json.load(f)

all_item_ids = list(item_index.keys())

cat_to_items = defaultdict(list)
for iid, meta in item_index.items():
    cat_to_items[meta['category']].append(iid)

print(f'Items totales    : {len(all_item_ids):,}')
print(f'Categorias unicas: {len(cat_to_items)}')


### 2. Generar tripletas

In [ ]:
from tqdm.notebook import tqdm

def sample_negative(anchor_id, outfit_set):
    for _ in range(30):
        candidate = random.choice(all_item_ids)
        if candidate not in outfit_set and candidate != anchor_id:
            return candidate
    return random.choice(all_item_ids)

def build_triplets(outfits, split_name):
    triplets = []
    for outfit in tqdm(outfits, desc=f'[{split_name}] outfits'):
        items = [it['item_id'] for it in outfit['items']]
        if len(items) < 2:
            continue
        outfit_set = set(items)

        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                anchor_id = items[i]
                pos_id    = items[j]

                # ✅ Negativo siempre de la misma categoría que el POSITIVO
                pos_cat = item_index.get(pos_id, {}).get('category', 'unknown')
                cands   = [c for c in cat_to_items.get(pos_cat, [])
                           if c not in outfit_set and c != anchor_id]

                for _ in range(N_NEG):
                    if cands:
                        neg_id = random.choice(cands)
                    else:
                        # Fallback — cualquier item fuera del outfit
                        neg_id = sample_negative(anchor_id, outfit_set)

                    triplets.append({
                        'anchor_id': anchor_id,
                        'pos_id'   : pos_id,
                        'neg_id'   : neg_id,
                    })

    random.shuffle(triplets)
    return triplets

triplets_by_split = {}
for split in ['train', 'valid']:
    with open(OUT_DIR / f'outfits_{split}.json') as f:
        outfits = json.load(f)
    print(f'\n[{split.upper()}] {len(outfits):,} outfits')
    triplets = build_triplets(outfits, split)
    triplets_by_split[split] = triplets
    with open(OUT_DIR / f'triplets_{split}_nondisjoint.json', 'w') as f:
        json.dump(triplets, f)
    print(f'  {len(triplets):,} tripletas guardadas -> triplets_{split}_nondisjoint .json')


In [ ]:
same_cat = sum(
    1 for t in triplets_by_split['train'][:5000]
    if item_index.get(str(t['pos_id']), {}).get('category') ==
       item_index.get(str(t['neg_id']), {}).get('category')
)
print(f'Pos y neg misma categoría: {same_cat}/5000 ({same_cat/50:.1f}%)')

### 3. Estadisticas de las tripletas

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, split in zip(axes, ['train', 'valid']):
    triplets = triplets_by_split[split]
    cats  = [item_index.get(t['anchor_id'], {}).get('category', '?') for t in triplets]
    cnt   = Counter(cats)
    top10 = cnt.most_common(10)
    labels, values = zip(*top10)
    ax.barh(list(labels)[::-1], list(values)[::-1], color='#4C72B0', edgecolor='white')
    ax.set_title(f'{split.capitalize()} -- {len(triplets):,} tripletas')
    ax.set_xlabel('Numero de tripletas')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Distribucion de categorias de anchors', fontsize=13)
plt.tight_layout()
plt.show()


### 4. Verificar cobertura con Fisher Vectors

In [ ]:
fv_path  = OUT_DIR / 'fisher_vectors_nondisjoint.npy'
ids_path = OUT_DIR / 'fisher_item_ids_nondisjoint.json'

if fv_path.exists() and ids_path.exists():
    fv_matrix = np.load(fv_path)
    with open(ids_path) as f:
        fv_ids = json.load(f)
    fv_lookup = {iid: idx for idx, iid in enumerate(fv_ids)}

    for split in ['train', 'valid']:
        triplets = triplets_by_split[split]
        n_cov = sum(
            1 for t in triplets
            if t['anchor_id'] in fv_lookup
            and t['pos_id']   in fv_lookup
            and t['neg_id']   in fv_lookup
        )
        pct  = 100 * n_cov / len(triplets)
        mark = 'OK' if pct > 95 else 'AVISO'
        print(f'  [{mark}] [{split}] Cobertura FV: {n_cov:,}/{len(triplets):,} ({pct:.1f}%)')

    print(f'\n  Fisher Vectors shape: {fv_matrix.shape}')
else:
    print('fisher_vectors_nondisjoint.npy no encontrado.')
    print('Ejecuta primero 02b_fisher_text_embeddings.ipynb')


### 5. Visualizar ejemplos de tripletas (imagenes)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

DATA_ROOT = Path('./polyvore_kaggle/polyvore_outfits')

def show_triplet(triplet, item_index, data_root):
    fig, axes = plt.subplots(1, 3, figsize=(10, 4))
    roles = [
        ('anchor',     triplet['anchor_id'], '#4C72B0'),
        ('positive',   triplet['pos_id'],    '#55A868'),
        ('negative',   triplet['neg_id'],    '#C44E52'),
    ]
    for ax, (role, iid, color) in zip(axes, roles):
        meta     = item_index.get(iid, {})
        img_path = data_root / meta.get('image_path', '')
        try:
            ax.imshow(mpimg.imread(img_path))
        except:
            ax.text(0.5, 0.5, 'Sin imagen', ha='center', va='center',
                    transform=ax.transAxes, fontsize=12)
        cat  = meta.get('category', '?')
        text = meta.get('text', '')[:40]
        ax.set_title(f'{role}\n{cat}\n{text}', fontsize=9, color=color, fontweight='bold')
        ax.axis('off')
    plt.suptitle('Ejemplo de tripleta', fontsize=12)
    plt.tight_layout()
    plt.show()

triplets_train = triplets_by_split['train']
for _ in range(2):
    idx = random.randint(0, len(triplets_train) - 1)
    show_triplet(triplets_train[idx], item_index, DATA_ROOT)


### 6. Resumen final

In [ ]:
print('Pipeline de datos completado\n')
print('Archivos en ./processed/:')

files = [
    ('outfits_train.json',   'outfits filtrados (<= 8 prendas)'),
    ('outfits_valid.json',   ''),
    ('outfits_test.json',    ''),
    ('item_index_full_nondisjoint.json',      'indice global de items'),
    ('triplets_train_nondisjoint.json',  'tripletas de entrenamiento'),
    ('triplets_valid_nondisjoint.json',  'tripletas de validacion'),
    ('fisher_vectors_nondisjoint.npy',   'embeddings texto Fisher HGLMM'),
    ('fisher_item_ids_nondisjoint.json', 'ids en el mismo orden'),
    ('gmm_model_nondisjoint.pkl',        'modelo GMM'),
    ('pca_model_nondisjoint.pkl',        'modelo PCA'),
    ('text_stats_nondisjoint.json',      'estadisticas del pipeline'),
]
for fname, desc in files:
    p    = OUT_DIR / fname
    mark = 'OK'      if p.exists() else 'PENDIENTE'
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else ''
    print(f'  {mark:10s}  {fname:<30}  {size:<12}  {desc}')

print('\nSiguiente: 03_sce_net_model.ipynb')
